In [15]:
import os
from tqdm import tqdm
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline 
import numpy as np
import random
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
def get_ids(ids, device, assistant_idx=32001):
    ids = ids.to(device)
    inputs = ids[1:].clone().to(device)
    labels = ids[:-1].clone().to(device)
    assistant_idx = torch.where(labels ==assistant_idx )[0]
    labels[:assistant_idx+1] = -100
    return inputs.unsqueeze(0), labels


# Constants

In [3]:
def file_to_string(filename):
    with open(filename, 'r') as file:
        return file.read()
EUREKA_ROOT_DIR="."

prompt_dir = f'./prompts'
initial_system = file_to_string(f'{prompt_dir}/initial_system.txt')
code_output_tip = file_to_string(f'{prompt_dir}/code_output_tip.txt')
initial_user = file_to_string(f'{prompt_dir}/initial_user.txt')
reward_signature = file_to_string(f'{prompt_dir}/reward_signature.txt')

tasks = {"acrobot":"to apply torques on the actuated joint to swing the free end of the linear chain above a given height from an initial state hanging downwards",
         "cartpolev1" : "to balance a pole on a cart so that the pole stays between +/- 0.209 away from the vertical",
        "cartpolev2" : "to balance a pole on a cart so that the pole stays between [0,0.21] away from the vertical",
        "orbit_discrete": "to accelerate the space shuttle to orbit the earth counter clockwise between the 2 specified bounds",
        "mountain_car": "The goal is to strategically accelerate the car to reach the goal state on top of the right hill."}

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
assistant_idx = 32001

In [4]:

model = AutoModelForCausalLM.from_pretrained( 
    "microsoft/Phi-3-mini-128k-instruct",  
    device_map="auto",  
    torch_dtype="auto",  
    trust_remote_code=True,  
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct") 
model
pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
) 

generation_args = { 
    "max_new_tokens": 4096, 
    "temperature": 1.0, 
    "do_sample": False, 
} 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
def generate_text(initial_system,initial_user, model):
    messages = [
    {
        "role": "system",
        "content": initial_system,
    },
    {"role": "user", "content": initial_user},
    ]
    with torch.no_grad():
        tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
        gen = model.generate(input_ids = tokenized_chat,**generation_args)[0].to("cpu")
        generated_text = tokenizer.decode(gen)
        del tokenized_chat
        torch.cuda.empty_cache()
    return generated_text, gen

In [25]:
def get_messages(task, description):
    env_name = task.replace("v1", "").replace("v2", "")
    if f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/classic_control'):
        env_parent = 'classic_control'  
    elif f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/augmented') :
        env_parent = 'augmented'
    elif f'{env_name}.py' in os.listdir(f'{EUREKA_ROOT_DIR}/envs/designed') :
        env_parent = 'designed'
    else : raise ValueError(f"Environment {env_name} not found in classic_control, designed or augmented directories!")
    task_obs_file = f'{EUREKA_ROOT_DIR}/envs/{env_parent}/{env_name}_obs.py'
    task_obs_code_string  = file_to_string(task_obs_file)
    
    system = initial_system.format(task_reward_signature_string=reward_signature)
    user = initial_user.format(task_obs_code_string=task_obs_code_string, task_description=description)
    return system, user

# Normal

In [6]:
generations = {}
for task, description in tqdm(tasks.items()):
    generations[task] = {}

    system, user = get_messages(task, description)

    generations[task]["text"], generations[task]["ids"] = generate_text(system, user,model)
torch.save(generations, f'./outputs/head_ablations/generations.pth')


  0%|          | 0/5 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
You are not running the flash-attention implementation, expect numerical differences.
100%|██████████| 5/5 [02:26<00:00, 29.25s/it]


## Generations

In [7]:
from IPython.display import clear_output, display, Markdown
# File to read from and the new file to write annotations t            
# Display the function content
for task,gen in generations.items():
    display(Markdown(f"## {task}\n```\n{gen['text'].split('<|assistant|>')[-1]}\n```"))

## acrobot
```
 To create a reward function for the AcrobotEnv environment that encourages the agent to swing the free end of the linear chain above a given height, we need to define a reward that increases as the end of the chain gets higher and decreases as it gets lower. We can use the angle of the second joint (which is the free end of the chain) as a measure of height. The goal is to maximize this angle while minimizing the energy used, which can be represented by the magnitude of the torque applied.

Here's a possible reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray, torque: np.ndarray) -> Tuple[float, Dict[str, float]]:
    """
    Reward function for AcrobotEnv.

    Args:
    - object_pos: np.ndarray, the current state of the environment (angles and angular velocities).
    - goal_pos: np.ndarray, the desired state of the environment (angles and angular velocities).
    - torque: np.ndarray, the torque applied to the second joint.

    Returns:
    - reward: float, the reward for the current state.
    - info: Dict[str, float], additional information (e.g., energy used).
    """
    # Extract the relevant state variables
    s0, s1, s2, s3 = object_pos
    goal_s0, goal_s1, goal_s2, goal_s3 = goal_pos

    # Calculate the height of the free end of the chain (second joint)
    # Assuming the length of the chain is 1 unit for simplicity
    chain_length = 1.0
    height = chain_length * (1 - cos(s2))

    # Calculate the desired height
    desired_height = chain_length * (1 - cos(goal_s2))

    # Calculate the reward based on the height difference
    height_reward = desired_height - height

    # Penalize the energy used (magnitude of the torque)
    energy_penalty = np.linalg.norm(torque)

    # Combine the height reward and energy penalty
    reward = height_reward - energy_penalty

    # Additional information about the energy used
    info = {'energy_used': energy_penalty}

    return reward, info
```

In this reward function, we calculate the height of the free end of the chain using the cosine of the second joint's angle. The reward is the difference between the desired height and the current height, which encourages the agent to swing the chain higher. We also penalize the energy used by the magnitude of the torque applied to the second joint, which encourages the agent to use less energy.

Please note that the actual implementation may require adjustments based on the specific dynamics of the AcrobotEnv environment and the desired behavior of the agent. The length of the chain and the relationship between the joint angles and the height of the free end may need to be calibrated based on the environment's specifications.<|end|>
```

## cartpolev1
```
 To create a reward function for the CartPole environment that encourages the pole to stay balanced within a specific angle range, we can use the absolute value of the pole's angle from the vertical as a measure of how far off-balance the pole is. We want to penalize the agent for the pole's angle deviating from the vertical by more than 0.209 radians (which is approximately 12 degrees).

Here's a possible implementation of the reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray) -> Tuple[float, Dict[str, float]]:
    # Extract the angle of the pole from the environment state
    _, _, theta, _ = object_pos
    
    # Define the acceptable angle range (in radians)
    acceptable_angle_range = 0.209
    
    # Calculate the reward as the negative of the absolute difference between the pole's angle and the acceptable range
    # This encourages the agent to keep the pole's angle within the range
    reward = -np.abs(theta - acceptable_angle_range)
    
    # Additional information can be included in the info dictionary
    info = {}
    
    return reward, info
```

In this implementation, the `object_pos` variable represents the current state of the environment, which includes the angle of the pole (`theta`). The `goal_pos` variable is not used in this reward function, but it can be included in the `info` dictionary if you want to provide additional information to the agent.

The reward function penalizes the agent for the pole's angle deviating from the acceptable range by using the negative absolute difference between the pole's angle and the acceptable range. This encourages the agent to keep the pole's angle within the range of +/- 0.209 radians from the vertical.

Note that this reward function only considers the angle of the pole and does not take into account the position of the cart or the velocity of the pole. If you want to include these factors in the reward function, you can modify the implementation accordingly.<|end|>
```

## cartpolev2
```
 To create a reward function for the CartPole environment that encourages the pole to stay within the specified range of angles from the vertical, we can use the following reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(state: np.ndarray, goal_angle: float = 0.21) -> Tuple[float, Dict[str, float]]:
    """
    Compute the reward for the CartPole environment.

    Args:
    - state (np.ndarray): The current state of the environment, containing the cart position, cart velocity, pole angle, and pole angular velocity.
    - goal_angle (float): The goal angle for the pole to stay within this range from the vertical.

    Returns:
    - Tuple[float, Dict[str, float]]: A tuple containing the reward and a dictionary of additional information.
    """
    # Unpack the state
    x, x_dot, theta, theta_dot = state

    # Calculate the reward
    # The reward is high if the pole is within the goal angle range, and low otherwise
    reward = 1.0 if abs(theta) <= goal_angle else 0.0

    # Additional information
    info = {
        'theta': theta,
        'theta_dot': theta_dot,
        'reward': reward
    }

    return reward, info
```

In this reward function, we calculate the reward based on the angle of the pole (`theta`). If the absolute value of `theta` is less than or equal to the goal angle (0.21 radians), we give a reward of 1.0, indicating that the pole is within the desired range. Otherwise, we give a reward of 0.0, indicating that the pole is not within the desired range.

The additional information dictionary contains the current angle of the pole (`theta`), the angular velocity of the pole (`theta_dot`), and the reward received. This information can be used for debugging or analysis purposes.

Note that this reward function assumes that the input `state` is a NumPy array with the following structure:

```
state = np.array([x, x_dot, theta, theta_dot])
```

where `x` is the horizontal position of the cart, `x_dot` is the velocity of the cart, `theta` is the angle of the pole, and `theta_dot` is the angular velocity of the pole.

Also, make sure to call the `compute_reward` function with the same device as the input tensors (e.g., CPU or GPU) to avoid any issues with tensor operations.<|end|>
```

## orbit_discrete
```
 To create a reward function for the OrbitEnv environment that encourages the space shuttle to accelerate to orbit the Earth counter-clockwise between the specified bounds, we need to consider several factors:

1. The shuttle's altitude (distance from the Earth's surface).
2. The shuttle's velocity (speed).
3. The shuttle's position relative to the desired orbit.

The reward function should encourage the shuttle to reach a stable orbit while staying within the specified bounds. We can use the following reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(state: np.ndarray, goal_pos: np.ndarray, goal_vel: np.ndarray) -> Tuple[float, Dict[str, float]]:
    # Extract the relevant state variables
    position = state[:2]
    velocity = state[2:]

    # Calculate the distance from the current position to the goal position
    distance_to_goal = np.linalg.norm(goal_pos - position)

    # Calculate the difference between the current velocity and the goal velocity
    velocity_diff = np.linalg.norm(goal_vel - velocity)

    # Calculate the reward based on the distance to the goal and the velocity difference
    # We want to minimize the distance and velocity difference, so we use negative values
    reward = -distance_to_goal - velocity_diff

    # Calculate additional information to return
    info = {
        'distance_to_goal': distance_to_goal,
        'velocity_diff': velocity_diff,
    }

    return reward, info
```

In this reward function, we use negative values for the distance to the goal and the velocity difference to encourage the shuttle to get closer to the goal position and match the desired velocity. The closer the shuttle is to the goal position and the closer its velocity is to the goal velocity, the higher the reward will be.

To use this reward function in the OrbitEnv environment, you would need to define the goal position and velocity for the desired orbit. For example, if the desired orbit is a counter-clockwise circle between the lower and upper bounds, you could define the goal position and velocity as follows:

```python
import numpy as np

# Define the desired orbit parameters
lower_bound = 5
upper_bound = 5.2
radius = (lower_bound + upper_bound) * OrbitEnv.R / 2

# Define the goal position and velocity for the desired orbit
goal_pos = np.array([radius * np.cos(np.pi / 4), radius * np.sin(np.pi / 4)])
goal_vel = np.array([0, np.sqrt(OrbitEnv.R * OrbitEnv.G / radius)])
```

Then, you would pass these goal position and velocity values to the reward function when calling it in the OrbitEnv environment.<|end|>
```

## mountain_car
```
 To create a reward function for the MountainCarEnv environment that encourages the agent to reach the goal state on top of the right hill, we need to consider the following aspects:

1. The car should be rewarded for reaching the goal state.
2. The car should be penalized for not reaching the goal state within a certain number of steps.
3. The car should be penalized for taking too long to reach the goal state.

Here's a possible implementation of the reward function:

```python
import numpy as np
from typing import Tuple, Dict

def compute_reward(object_pos: np.ndarray, goal_pos: np.ndarray, max_steps: int = 200) -> Tuple[float, Dict[str, float]]:
    """
    Compute the reward for the MountainCarEnv environment.

    Args:
        object_pos (np.ndarray): The current position of the car on the x axis.
        goal_pos (np.ndarray): The goal position of the car on the x axis.
        max_steps (int): The maximum number of steps allowed to reach the goal state.

    Returns:
        Tuple[float, Dict[str, float]]: A tuple containing the reward and a dictionary of additional information.
    """
    # Calculate the distance between the current position and the goal position
    distance_to_goal = np.abs(goal_pos - object_pos)

    # Reward for reaching the goal state
    if distance_to_goal < 0.1:  # Assuming the goal state is within 0.1 units of the goal position
        reward = 1.0
        info = {'success': True}
    else:
        # Penalize for not reaching the goal state
        reward = -1.0
        info = {'success': False}

    # Penalize for taking too many steps
    if max_steps - len(info['success']) < 0:
        reward -= 0.1
        info['penalty_steps'] = True

    return reward, info
```

In this implementation, the reward function rewards the agent for reaching the goal state (within a certain threshold) and penalizes it for not reaching the goal state or taking too many steps. The `info` dictionary contains additional information about the success of the agent and whether it took too many steps.

You can adjust the reward and penalty values according to your specific needs. Additionally, you can modify the `max_steps` parameter to control the maximum number of steps allowed to reach the goal state.<|end|>
```

## Perplexities

In [6]:
generations = torch.load(f'./outputs/head_ablations/generations.pth')

/tmp/ipykernel_3740765/253827073.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generations = torch.load(f'./outputs/head_ablations/generations.pth')


In [7]:
losses = {}
with torch.no_grad(): 
    for task in generations.keys():

        inputs,labels = get_ids(generations[task]["ids"], model.device)
        output = model(input_ids=inputs, labels=labels)
        losses[task] = output.loss
        print(task,f"size:{inputs.shape}",f"loss:{output.loss}")

You are not running the flash-attention implementation, expect numerical differences.


acrobot size:torch.Size([1, 1069]) loss:21.87228775024414
cartpolev1 size:torch.Size([1, 836]) loss:18.808198928833008
cartpolev2 size:torch.Size([1, 945]) loss:18.73197364807129
orbit_discrete size:torch.Size([1, 1024]) loss:19.83180809020996
mountain_car size:torch.Size([1, 925]) loss:18.877521514892578


# Modifying PHI3 attention

In [8]:
from utils import get_ablated_model
import warnings
from tqdm import tqdm

In [9]:
model = get_ablated_model(model)

In [10]:
num_layers, num_heads = model.config.num_hidden_layers, model.config.num_attention_heads

In [11]:
# Suppress all warnings
warnings.filterwarnings("ignore")
perplexities  = {task:torch.zeros((num_layers, num_heads)).to(model.device) for task in tasks.keys()}
with torch.no_grad():
    for task in tasks.keys():
        inputs, labels = get_ids(generations[task]["ids"], model.device)
        for l in tqdm(range(num_layers)):
            model.model.layers._modules[str(l)].self_attn.ablate = True
            for h in range(num_heads):
                model.model.layers._modules[str(l)].self_attn.ablate_idx = [h]
                output = model(input_ids=inputs, labels=labels)
                perplexities[task][l,h] = output.loss
        
            model.model.layers._modules[str(l)].self_attn.ablate = False
torch.save(perplexities, f'./outputs/head_ablations/perplexities.pth')


100%|██████████| 32/32 [04:15<00:00,  7.99s/it]


In [41]:
def get_heatmap(perplexities, task):
    # Sample NumPy matrix
    matrix = perplexities.cpu().numpy()# Create the heatmap
    heatmap = go.Figure(data=go.Heatmap(
        z=matrix,
        x=np.arange(matrix.shape[1]),  # Column indices for x-axis
        y=np.arange(matrix.shape[0]),  # Row indices for y-axis
        colorscale='Magma',  # Set the color scale to 'Magma'
    ))

    # Add axis titles
    heatmap.update_layout(
        title=f'Heatmap for {task}',
        xaxis_title='Attention Heads',
        yaxis_title='Layers'
    )

    # Show the heatmap
    return heatmap


In [42]:
for task in tasks.keys():
    heatmap = get_heatmap(perplexities[task], task)
    pio.write_html(heatmap, f"./outputs/head_ablations/{task}_heatmap.html")

In [43]:
perplexities_quantiles = {}
for task in tasks.keys():

    # Example tensor of shape (32, 32)
    tensor = perplexities[task]

    # Find the 99th percentile value
    percentile_value = torch.quantile(tensor, 0.99)

    # Create a mask that keeps values above the 99th percentile
    mask = tensor >= percentile_value

    # Apply the mask to the tensor
    masked_tensor = tensor * mask
    masked_tensor/= masked_tensor.sum()
    perplexities_quantiles[task] = masked_tensor
    heatmap = get_heatmap(perplexities_quantiles[task], task)
    pio.write_html(heatmap, f"./outputs/head_ablations/{task}_heatmap_quantiles.html")
